In [0]:
# ================================================
# READ FROM INGESTION OUTPUT
# ================================================
from pyspark.sql.functions import (
    col, count, when, trim, to_date,
    year, month, quarter, date_format,
    row_number
)
from pyspark.sql.window import Window

df_raw = spark.table("default.nottinghamshire_raw")

print(f"✅ Rows loaded: {df_raw.count():,}")
print(f"✅ Columns: {df_raw.columns}")

In [0]:
# ================================================
# DATA QUALITY ASSESSMENT
# ================================================

print("=== NULL COUNTS PER COLUMN ===")
null_counts = df_raw.select([
    count(when(
        col(c).isNull() | (col(c) == ""), c)
    ).alias(c)
    for c in df_raw.columns
])
display(null_counts)

print("\n=== UNIQUE CRIME TYPES ===")
df_raw.groupBy("crime_type") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(20, truncate=False)

print("\n=== NO LOCATION ROWS ===")
no_location = df_raw.filter(
    col("location") == "No Location"
).count()
total = df_raw.count()
print(f"No location rows: {no_location:,}")
print(f"Percentage of total: {round(no_location/total*100, 2)}%")

In [0]:
# ================================================
# CLEANING LAYER
# ================================================

rows_before = df_raw.count()
print(f"Rows BEFORE cleaning: {rows_before:,}")

df_clean = df_raw \
    .drop("context", "falls_within") \
    .withColumn("crime_type",
        trim(col("crime_type"))) \
    .withColumn("outcome",
        trim(col("outcome"))) \
    .withColumn("force_name",
        trim(col("force_name"))) \
    .withColumn("year",
        year(to_date(col("month_year"), "yyyy-MM"))) \
    .withColumn("month_num",
        month(to_date(col("month_year"), "yyyy-MM"))) \
    .withColumn("quarter",
        quarter(to_date(col("month_year"), "yyyy-MM"))) \
    .withColumn("month_name",
        date_format(
            to_date(col("month_year"), "yyyy-MM"), "MMMM"
        )) \
    .withColumn("has_location",
        when(col("location") == "No Location", False)
        .otherwise(True)) \
    .withColumn("outcome",
        when(col("outcome").isNull(), "Not Recorded")
        .otherwise(col("outcome"))) \
    .withColumn("crime_id",
        when(col("crime_id") == "", None)
        .otherwise(col("crime_id")))

rows_after_cleaning = df_clean.count()
print(f"Rows AFTER cleaning: {rows_after_cleaning:,}")
print(f"Difference: {rows_before - rows_after_cleaning:,}")

In [0]:
# ================================================
# FILTER TO NOTTINGHAMSHIRE AREAS ONLY
# Remove cross boundary crimes and no location
# Documented as known data quality limitation
# ================================================

rows_before_filter = df_clean.count()

nottinghamshire_areas = [
    "Nottingham",
    "Ashfield",
    "Mansfield",
    "Newark and Sherwood",
    "Bassetlaw",
    "Broxtowe",
    "Gedling",
    "Rushcliffe"
]

pattern = "|".join(nottinghamshire_areas)

df_clean = df_clean \
    .filter(col("lsoa_code").isNotNull()) \
    .filter(col("lsoa_name").isNotNull()) \
    .filter(col("location") != "No Location") \
    .filter(col("lsoa_name").rlike(pattern))

rows_after_filter = df_clean.count()

print(f"Rows before filter: {rows_before_filter:,}")
print(f"Rows after filter: {rows_after_filter:,}")
print(f"Rows removed: {rows_before_filter - rows_after_filter:,}")
print(f"Percentage removed: {round((rows_before_filter - rows_after_filter)/rows_before_filter*100, 2)}%")

In [0]:
# ================================================
# DEDUPLICATION
# Only applied to records WITH a Crime ID
# ASB null IDs preserved untouched
# ================================================

df_null_ids = df_clean.filter(col("crime_id").isNull())
df_with_ids = df_clean.filter(col("crime_id").isNotNull())

print(f"Records with no Crime ID (ASB): {df_null_ids.count():,}")
print(f"Records with Crime ID: {df_with_ids.count():,}")

df_prioritised = df_with_ids.withColumn(
    "outcome_priority",
    when(col("outcome") == "Offender sent to prison", 1)
    .when(col("outcome") == "Offender given a caution", 2)
    .when(col("outcome") == "Court result unavailable", 3)
    .when(col("outcome") == "Local resolution", 4)
    .when(col("outcome") == "Unable to prosecute suspect", 5)
    .when(col("outcome") == "Investigation complete; no suspect identified", 6)
    .when(col("outcome") == "Action to be taken by another organisation", 7)
    .when(col("outcome") == "Further action is not in the public interest", 8)
    .when(col("outcome") == "Formal action is not in the public interest", 9)
    .when(col("outcome") == "Further investigation is not in the public interest", 10)
    .when(col("outcome") == "Status update unavailable", 11)
    .otherwise(12)
)

window_spec = Window \
    .partitionBy("crime_id", "lsoa_code") \
    .orderBy("outcome_priority")

df_with_ids_deduped = df_prioritised \
    .withColumn("rank", row_number().over(window_spec)) \
    .filter(col("rank") == 1) \
    .drop("rank", "outcome_priority")

df_final_clean = df_null_ids.union(df_with_ids_deduped)

print(f"\n✅ DEDUPLICATION COMPLETE")
print(f"Rows before dedup: {df_clean.count():,}")
print(f"Rows after dedup: {df_final_clean.count():,}")
print(f"Rows removed: {df_clean.count() - df_final_clean.count():,}")

remaining_dupes = df_final_clean \
    .filter(col("crime_id").isNotNull()) \
    .groupBy("crime_id", "lsoa_code") \
    .count() \
    .filter(col("count") > 1)
print(f"Remaining duplicates: {remaining_dupes.count()}")

In [0]:
# ================================================
# VALIDATION LAYER
# ================================================

print("=== NULL CHECK ON KEY REPORTING FIELDS ===")
key_fields = [
    "force_name", "year", "quarter",
    "month_num", "month_name", "crime_type",
    "lsoa_code", "lsoa_name"
]
for field in key_fields:
    null_count = df_final_clean.filter(
        col(field).isNull()
    ).count()
    status = "✅" if null_count == 0 else "⚠️"
    print(f"{status} {field}: {null_count} nulls")

print("\n=== YEAR RANGE CHECK ===")
df_final_clean.groupBy("year") \
    .count() \
    .orderBy("year") \
    .show()

print("\n=== MONTH COVERAGE CHECK ===")
df_final_clean.groupBy("year", "month_num") \
    .count() \
    .orderBy("year", "month_num") \
    .show(100)

print("\n=== CRIME TYPE CHECK ===")
df_final_clean.groupBy("crime_type") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(truncate=False)

In [0]:
# ================================================
# SAVE CLEANED DATA
# ================================================

df_final_clean.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("default.nottinghamshire_clean")

print("✅ Cleaned data saved to default.nottinghamshire_clean")
print(f"✅ Total rows saved: {df_final_clean.count():,}")
print(f"✅ Columns: {df_final_clean.columns}")